# Las fuentes

Este pipeline sólo baja los datos crudos a `data/raw` sin
procesarlos, todo el procesamiento está en el 1_.

Cada crudo está en el repo en un zip al lado de su archivo, y este pipeline lo descomprime. Sólo
baja de la fuente lo que no tiene zip. Wikidata y OpenStreetMap son bases de datos activas: para
traerlas de nuevo hay que borrar su zip y su archivo, y los resultados pueden cambiar un poco.

In [16]:
%pip install xlrd geopandas pyarrow scipy


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import json, time, zipfile
from pathlib import Path
import requests
import urllib3

# El GeoServer del IDE tiene el certificado vencido: se baja con verify=False.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

RAW = Path('data/raw')
RAW.mkdir(parents=True, exist_ok=True)

UA = {'User-Agent': 'InfoTrenes (contacto: github.com/yi-yi-dev)'}
WFS = ('https://ide.transporte.gob.ar/geoserver/idera/ows'
       '?service=WFS&version=1.0.0&request=GetFeature'
       '&outputFormat=application%2Fjson&typeName=idera:')
# Overpass a veces está saturado y responde 504, así que se prueba en otros servidores.
SERVIDORES_OVERPASS = ['https://overpass-api.de/api/interpreter',
                       'https://overpass.kumi.systems/api/interpreter',
                       'https://overpass.private.coffee/api/interpreter']

def descomprimir(crudo):
    # Va al repo el zip y no el archivo, porque GitHub no acepta archivos de más de 100 MB.
    zipeado = RAW / f'{crudo.name}.zip'
    if not crudo.exists() and zipeado.exists():
        with zipfile.ZipFile(zipeado) as z:
            z.extract(crudo.name, RAW)
    return crudo.exists()

def guardar(crudo, contenido):
    crudo.write_bytes(contenido)
    with zipfile.ZipFile(RAW / f'{crudo.name}.zip', 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        z.write(crudo, crudo.name)

def overpass(consulta, archivo):
    crudo = RAW / archivo
    if not descomprimir(crudo):
        for servidor in SERVIDORES_OVERPASS * 2:
            r = requests.post(servidor, data={'data': consulta}, headers=UA, timeout=10000)
            if r.status_code == 200:
                break
            print(servidor, r.status_code)
            time.sleep(30)
        r.raise_for_status()
        guardar(crudo, r.content)
    print(archivo, len(json.loads(crudo.read_bytes())['elements']), 'elementos')

## Dataset 1 - ondat_ffcc_hist.xls

FUENTE: ONDaT (Observatorio Nacional de Datos de Transporte), C3T (Centro Tecnológico de
Transporte, Tránsito y Seguridad Vial), UTN. Un .xls con una planilla por año, 1857–2015.

- Las primeras 13 filas describen el archivo; las dos primeras columnas son el año y los km de red
  en servicio. Tiene años sin dato y la serie termina en 2000.

In [18]:
URL = ('https://ondat.c3t-utn.ar/wp-content/uploads/2015/08/'
       'Transporte-Ferroviario.-Datos-generales-hist%C3%B3ricos-062015.xls')
crudo = RAW / 'ondat_ffcc_hist.xls'
if not descomprimir(crudo):
    r = requests.get(URL, timeout=60)
    r.raise_for_status()
    guardar(crudo, r.content)
print(f'{crudo.stat().st_size:,} bytes')

55,296 bytes


## Dataset 2 - wikidata_estaciones.csv

FUENTE: Wikidata Query Service, endpoint SPARQL público, Wikimedia Foundation. Es la única fuente de
fechas de todo el proyecto.

- **st** — URI de la estación. Puede aparecer en más de una fila, así que hay que deduplicar por acá.
- **stLabel** — Nombre de la estación.
- **coord** — Coordenadas en WKT, `Point(lon lat)`.
- **opened** — P1619, apertura al público. Es el año de alta.
- **inception** — P571, construcción. Año de alta de respaldo cuando falta `opened`.
- **closed** — P3999, cierre oficial. Es el año de baja.

In [19]:
QUERY = """
SELECT ?st ?stLabel ?coord ?opened ?closed ?inception WHERE {
  ?st wdt:P31/wdt:P279* wd:Q55488 .   # es una estación de tren (o subclase)
  ?st wdt:P17 wd:Q414 .               # país = Argentina
  OPTIONAL { ?st wdt:P625  ?coord }     # coordenadas
  OPTIONAL { ?st wdt:P1619 ?opened }    # apertura al público
  OPTIONAL { ?st wdt:P3999 ?closed }    # cierre oficial
  OPTIONAL { ?st wdt:P571  ?inception } # construcción
  SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}
"""
crudo = RAW / 'wikidata_estaciones.csv'
if not descomprimir(crudo):
    r = requests.get('https://query.wikidata.org/sparql', params={'query': QUERY},
                     headers={**UA, 'Accept': 'text/csv'}, timeout=600)
    r.raise_for_status()
    guardar(crudo, r.text.encode('utf-8'))
# Wikidata devuelve respuestas truncadas con HTTP 200, así que se valida el tamaño.
print(crudo.read_text(encoding='utf-8').count('\n') - 1, 'filas')

3196 filas


## Dataset 3 - Ferrocarril.geojson y Estacion_Ferroviaria.geojson

FUENTE: IDE de la Secretaría de Transporte de la Nación, GeoServer público, workspace `idera`, por
WFS. Las dos capas vienen en EPSG:4326.

- **Ferrocarril.geojson** — 318 tramos MultiLineString, la traza física. Usa `FUN` (`Activo` /
  `No operativo`), única fuente del estado actual, y `kms`, el largo declarado.
- **Estacion_Ferroviaria.geojson** — las estaciones construidas: `geometry` y `nombre`.

In [20]:
for capa in ['Ferrocarril', 'Estacion_Ferroviaria']:
    crudo = RAW / f'{capa}.geojson'
    if not descomprimir(crudo):
        r = requests.get(WFS + capa, timeout=300, verify=False)
        r.raise_for_status()
        guardar(crudo, r.content)
    print(f'{capa:<22} {crudo.stat().st_size:>12,} bytes')

Ferrocarril               4,424,731 bytes
Estacion_Ferroviaria        933,379 bytes


## Dataset 4 - osm_rail_abandoned.json

FUENTE: OpenStreetMap vía la API Overpass, OSM Foundation, en su estado actual. Son los ways de vía
en desuso, abandonada, arrasada o desmantelada de la Argentina, en el JSON crudo de Overpass.

- De cada way sólo se usa `geometry`, la polilínea como lista de `{lat, lon}`: es la vía que ya no
  está en la capa del IDE. Sus etiquetas no se usan.

In [21]:
overpass("""
[out:json][timeout:10000];
area["ISO3166-1"="AR"][admin_level=2]->.a;
(
  way["railway"="abandoned"](area.a);
  way["railway"="disused"](area.a);
  way["railway"="razed"](area.a);
  way["railway"="dismantled"](area.a);
);
out geom;
""", 'osm_rail_abandoned.json')

https://overpass-api.de/api/interpreter 504
osm_rail_abandoned.json 9589 elementos


## Dataset 5 — osm_rutas_tren.json

FUENTE: OpenStreetMap vía la API Overpass, OSM Foundation, en su estado actual. Son las relaciones
`route=train` de un rectángulo que cubre la Argentina y partes de los países vecinos, en el JSON
crudo de Overpass.

- `tags.service` y `tags.public_transport:version` dicen si la ruta es un servicio de pasajeros.
- Los miembros de tipo way, con su `geometry`, son la vía que recorre ese tren.

In [22]:
overpass("""
[out:json][timeout:10000];
relation["route"="train"](-55.1,-73.6,-21.7,-53.6);
out geom;
""", 'osm_rutas_tren.json')

https://overpass-api.de/api/interpreter 504
https://overpass.kumi.systems/api/interpreter 504
https://overpass.private.coffee/api/interpreter 504
https://overpass-api.de/api/interpreter 504
osm_rutas_tren.json 147 elementos


## Dataset 6 — radios1991.csv, radios2001.csv, radios2010.csv y radios2022.csv

FUENTE: INDEC vía datos.gob.ar, el Portal Nacional de Datos Abiertos. Son cuatro CSV de 110 a 130 MB,
uno por censo, con el polígono de cada radio censal.

- **wkt** — el polígono del radio.
- **codigo** y **cod_prov** — el radio y su provincia; los cinco primeros dígitos del código son el
  departamento.
- La población, que cambia de nombre: `B_POB_TOT` en 1991 y 2010, `POB_TOT` en 2001 y `POB_TOT_P`
  en 2022, que es sólo la gente en viviendas particulares.

In [23]:
DISTRIBUCIONES = {1991: '50.10', 2001: '50.7', 2010: '50.4', 2022: '50.1'}   # en datos.gob.ar

for censo, distribucion in DISTRIBUCIONES.items():
    csv = RAW / f'radios{censo}.csv'
    if not descomprimir(csv):
        url = ('https://infra.datos.gob.ar/catalog/indec/dataset/50'
               f'/distribution/{distribucion}/download/radios-censales-{censo}.csv')
        r = requests.get(url, timeout=900)
        r.raise_for_status()
        guardar(csv, r.content)
    print(f'{csv.name}: {csv.stat().st_size / 1e6:.0f} MB')

radios1991.csv: 111 MB
radios2001.csv: 116 MB
radios2010.csv: 121 MB
radios2022.csv: 131 MB


## Dataset 7 — ign_provincia.json

FUENTE: IGN (Instituto Geográfico Nacional), capa `ign:provincia` del GeoServer público, por WFS. Es
el mapa base de todo lo que se dibuja sobre el país, y sólo se usa `geometry`. Pesa 111 MB porque
trae la costa al detalle.

In [24]:
URL = ('https://wms.ign.gob.ar/geoserver/ows?service=WFS&version=2.0.0'
       '&request=GetFeature&typeNames=ign:provincia&outputFormat=application/json')
crudo = RAW / 'ign_provincia.json'
if not descomprimir(crudo):
    r = requests.get(URL, timeout=900, verify=False)
    r.raise_for_status()
    guardar(crudo, r.content)
print(f'{crudo.name}: {crudo.stat().st_size / 1e6:.0f} MB')

ign_provincia.json: 112 MB
